# Phase 7: Custom Data Evaluation & Metrics Dashboard

This notebook loads the trained **Spiking Transformer U-Net** and evaluates it on custom data paths.
It reports **Dice Scores** and hardware-level **Agentic Routing Savings**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q snntorch nibabel transformers accelerate bitsandbytes


In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
class BraTS3DDataset(Dataset):
    def __init__(self, patient_dirs, patch_size=(64, 64, 64)):
        self.patient_dirs = patient_dirs
        self.patch_size = patch_size

    def __len__(self):
        return len(self.patient_dirs)

    def __getitem__(self, idx):
        patient_dir = self.patient_dirs[idx]
        t1n = nib.load(glob.glob(os.path.join(patient_dir, "*-t1n.nii*"))[0]).get_fdata().astype(np.float32)
        t1c = nib.load(glob.glob(os.path.join(patient_dir, "*-t1c.nii*"))[0]).get_fdata().astype(np.float32)
        t2w = nib.load(glob.glob(os.path.join(patient_dir, "*-t2w.nii*"))[0]).get_fdata().astype(np.float32)
        t2f = nib.load(glob.glob(os.path.join(patient_dir, "*-t2f.nii*"))[0]).get_fdata().astype(np.float32)
        seg = nib.load(glob.glob(os.path.join(patient_dir, "*-seg.nii*"))[0]).get_fdata().astype(np.float32)

        image_4d = np.transpose(np.stack([t1n, t1c, t2w, t2f], axis=0), (0, 3, 1, 2))
        seg_3d = np.transpose(seg, (2, 0, 1))

        d, h, w = image_4d.shape[1:]
        pd, ph, pw = self.patch_size
        if d < pd or h < ph or w < pw:
            image_4d = np.pad(image_4d, ((0,0), (0, max(0, pd-d)), (0, max(0, ph-h)), (0, max(0, pw-w))))
            seg_3d = np.pad(seg_3d, ((0, max(0, pd-d)), (0, max(0, ph-h)), (0, max(0, pw-w))))
            d, h, w = image_4d.shape[1:]

        z, y, x = np.random.randint(0, d-pd+1), np.random.randint(0, h-ph+1), np.random.randint(0, w-pw+1)
        image_patch = image_4d[:, z:z+pd, y:y+ph, x:x+pw]
        seg_patch = seg_3d[z:z+pd, y:y+ph, x:x+pw]

        mask_channels = np.stack([(seg_patch > 0), np.logical_or(seg_patch==1, seg_patch==3), (seg_patch==3)], axis=0).astype(np.float32)

        for c in range(4):
            img_c = image_patch[c]
            if img_c.max() > 0: image_patch[c] = (img_c - img_c[img_c>0].mean()) / (img_c[img_c>0].std() + 1e-8)

        return {'image': torch.from_numpy(image_patch), 'mask': torch.from_numpy(mask_channels)}

## 1. Load Architecture and Trained Weights


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/models/spiking_unet_best.pth"
CUSTOM_DATA_DIR = "/content/drive/MyDrive/datasets/custom_eval" # Update this path

# ... (Wait, I should define the model classes here again for a standalone notebook) ...
# (Including simplified versions of the classes for evaluation)
class TernarySurrogate(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, threshold=1.0):
        ctx.save_for_backward(input)
        ctx.threshold = threshold
        out = torch.zeros_like(input)
        out[input >= threshold], out[input <= -threshold] = 1.0, -1.0
        return out
    @staticmethod
    def backward(ctx, grad_out):
        input, = ctx.saved_tensors
        return grad_out * torch.exp(-torch.abs(input)), None

class TernaryLIF(nn.Module):
    def __init__(self, beta=0.9, threshold=1.0):
        super().__init__()
        self.beta, self.threshold = beta, threshold
        self.register_buffer("mem", None)
    def reset_state(self): self.mem = None
    def forward(self, x):
        if self.mem is None or self.mem.shape != x.shape: self.mem = torch.zeros_like(x)
        self.mem = self.beta * self.mem + x
        spk = TernarySurrogate.apply(self.mem, self.threshold)
        self.mem = self.mem - spk * self.threshold
        return spk


In [ ]:
# Boilerplate architecture (Tokenizer, Encoder, Decoder, AgenticLLM) matching 05_full_training.ipynb
# (Omitted here for brevity in the creation script, but will be full in actual notebook)
def shiftmax(x, dim=-1):
    x_norm = x - torch.max(x, dim=dim, keepdim=True)[0]
    approx_exp = 2.0 ** torch.floor(x_norm)
    return approx_exp / (torch.sum(approx_exp, dim=dim, keepdim=True) + 1e-8)

class BipolarSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.head_dim, self.num_heads = embed_dim // num_heads, num_heads
        self.q_proj, self.k_proj, self.v_proj, self.out_proj = [nn.Linear(embed_dim, embed_dim, bias=False) for _ in range(4)]
        self.lif_q, self.lif_k, self.lif_v = TernaryLIF(), TernaryLIF(), TernaryLIF()
    def reset_states(self): [l.reset_state() for l in [self.lif_q, self.lif_k, self.lif_v]]
    def forward(self, x):
        B, S, E = x.shape
        q_spk, k_spk, v_spk = self.lif_q(self.q_proj(x)), self.lif_k(self.k_proj(x)), self.lif_v(self.v_proj(x))
        q, k, v = [s.view(B, S, self.num_heads, self.head_dim).transpose(1, 2) for s in [q_spk, k_spk, v_spk]]
        attn = shiftmax(torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5), dim=-1)
        return self.out_proj(torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, S, E))

class SpikingTokenizer3D(nn.Module):
    def __init__(self, in_channels=4, embed_dim=32):
        super().__init__()
        self.conv1, self.conv2 = nn.Conv3d(in_channels, embed_dim//2, 3, 2, 1), nn.Conv3d(embed_dim//2, embed_dim, 3, 2, 1)
        self.lif1, self.lif2 = TernaryLIF(), TernaryLIF()
    def reset_states(self): self.lif1.reset_state(); self.lif2.reset_state()
    def forward(self, x):
        s1 = self.lif1(self.conv1(x))
        return self.lif2(self.conv2(s1)), s1

class SpikingTransformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, depth):
        super().__init__()
        self.layers = nn.ModuleList([BipolarSelfAttention(embed_dim, num_heads) for _ in range(depth)])
    def reset_states(self): [l.reset_states() for l in self.layers]
    def forward(self, x):
        B, E, D, H, W = x.shape
        x_flat = x.view(B, E, D*H*W).transpose(1, 2)
        for layer in self.layers: x_flat = x_flat + layer(x_flat)
        return x_flat.transpose(1, 2).view(B, E, D, H, W)

class SpikingUNetDecoder3D(nn.Module):
    def __init__(self, embed_dim, out_channels=3):
        super().__init__()
        self.upconv1, self.upconv2 = nn.ConvTranspose3d(embed_dim, embed_dim//2, 4, 2, 1), nn.ConvTranspose3d(embed_dim//2, out_channels, 4, 2, 1)
        self.lif_up1 = TernaryLIF()
    def reset_states(self): self.lif_up1.reset_state()
    def forward(self, encoder_spk, skip_spk):
        return self.upconv2(self.lif_up1(self.upconv1(encoder_spk) + skip_spk))


In [ ]:
class Phi3TriageAgent:
    def __init__(self):
        conf = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
        self.model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct", quantization_config=conf, device_map="auto", trust_remote_code=False)
        self.pipe = pipeline("text-generation", model=self.model, tokenizer=self.tokenizer)
    def decide_routing(self, entropy, fg):
        prompt = f"<|user|>Uncertainty: {entropy:.4f}. Foreground: {fg:.2f}. Route HIGH or LOW?<|end|><|assistant|>"
        out = self.pipe(prompt, max_new_tokens=5, temperature=0.1, do_sample=False)
        res = out[0]['generated_text'].upper()
        return "HIGH" in res

class AgenticSpikingTransformerLLM(nn.Module):
    def __init__(self, agent_llm, in_channels=4, out_channels=3, embed_dim=32, num_heads=4, depth=2):
        super().__init__()
        self.tokenizer, self.encoder, self.decoder, self.agent_llm = SpikingTokenizer3D(in_channels, embed_dim), SpikingTransformerEncoder(embed_dim, num_heads, depth), SpikingUNetDecoder3D(embed_dim, out_channels), agent_llm
        self.routing_stats = {'high': 0, 'low': 0}
    def reset_all_states(self): self.tokenizer.reset_states(); self.encoder.reset_states(); self.decoder.reset_states()
    def forward_step(self, x):
        enc_spk, skip_spk = self.tokenizer(x)
        return self.decoder(self.encoder(enc_spk), skip_spk)
    def forward(self, x):
        self.reset_all_states()
        mem = self.forward_step(x)
        p = torch.sigmoid(mem)
        ent = (-p * torch.log(p+1e-6) - (1-p)*torch.log(1-p+1e-6)).mean().item()
        route_high = self.agent_llm.decide_routing(ent, (p>0.5).sum().item())
        if route_high:
            self.routing_stats['high'] += 1
            for _ in range(3): mem += self.forward_step(x)
            return (mem / 4.0).unsqueeze(0)
        self.routing_stats['low'] += 1
        return mem.unsqueeze(0)


## 2. Evaluation Logic


In [ ]:
def dice_coef(p, t):
    p = (torch.sigmoid(p) > 0.5).float()
    return (2 * (p * t).sum() + 1e-5) / (p.sum() + t.sum() + 1e-5)

def run_evaluation(model, loader):
    model.eval()
    dices = []
    print("Starting Evaluation...")
    with torch.no_grad():
        for i, b in enumerate(loader):
            img, msk = b['image'].to(device), b['mask'].to(device)
            out = model(img) # Dynamic Routing
            d = dice_coef(out.squeeze(0), msk)
            dices.append(d.item())
            print(f"Volume {i+1} | Dice: {d.item():.4f}")
    
    avg_dice = np.mean(dices)
    savings = (model.routing_stats['low'] / (sum(model.routing_stats.values()) + 1e-8)) * 75.0 # Savings logic T1 vs T4
    print(f"\n--- RESULTS ---")
    print(f"Mean Dice Score: {avg_dice:.4f}")
    print(f"Agentic Savings (Energy/Time): {savings:.2f}%")
    print(f"Routing Ratio: {model.routing_stats}")


In [ ]:
# Setup Evaluation
dirs = [os.path.join(CUSTOM_DATA_DIR, d) for d in os.listdir(CUSTOM_DATA_DIR) if os.path.isdir(os.path.join(CUSTOM_DATA_DIR, d))]
if not dirs:
    print("No custom data found at: ", CUSTOM_DATA_DIR)
else:
    eval_ds = BraTS3DDataset(dirs)
    eval_ld = DataLoader(eval_ds, batch_size=1, shuffle=False)

    # Initialize Agent and Model
    agent = Phi3TriageAgent()
    model = AgenticSpikingTransformerLLM(agent).to(device)

    # Load Weights
    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
        print("Loaded model weights from:", MODEL_PATH)
        run_evaluation(model, eval_ld)
    else:
        print("Model file not found! Check path:", MODEL_PATH)